# ClinOntoScreen: NHANES + medical ontology + ML

This notebook downloads public NHANES XPT files, harmonizes the historical and later cycles, constructs a leakage-safe cohort, loads the OWL/RDF screening ontology, trains Logistic Regression and XGBoost models, evaluates ontology-assisted screening, and exports artifacts for the Streamlit dashboard.

**Primary question:** at the same screening capacity, does an ontology-assisted ranking identify more participants with HbA1c-defined screen-positive dysglycemia than XGBoost alone?

**Primary outcome:** among adults (18+) reporting neither prior diabetes nor prior prediabetes, `LBXGH >= 5.7%`. This is a one-measurement research label, not a confirmed clinical diagnosis.

**Data periods:** NHANES 2017–March 2020 for training/validation; NHANES August 2021–August 2023 is kept as the locked temporal test. CDC notes the later cycle has an updated sample design and modified questionnaire/examination procedures, so results are temporal generalization, not evidence of a specific cause of shift.

The notebook runs on CPU in Kaggle or Colab. Enable Internet for data downloads. It does not use the temporal test set for model selection. No performance result is assumed in advance.

In [ ]:
%pip -q install pandas numpy requests scikit-learn xgboost rdflib joblib matplotlib


In [ ]:
from __future__ import annotations
from io import BytesIO
from pathlib import Path
from datetime import datetime, timezone
import json, os, platform, warnings, zipfile, sys
from IPython.display import display
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from rdflib import Graph
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
import joblib

# Paths are selected for Kaggle, Colab, or a local Jupyter runtime.
if Path('/kaggle/working').exists() and ('KAGGLE_KERNEL_RUN_TYPE' in os.environ or Path('/kaggle/input').exists()):
    WORK_DIR = Path('/kaggle/working/clinontoscreen_run')
elif 'google.colab' in sys.modules:
    WORK_DIR = Path('/content/clinontoscreen_run')
else:
    WORK_DIR = Path.cwd() / 'clinontoscreen_run'
CACHE_DIR = WORK_DIR / 'data_cache'
ARTIFACT_DIR = WORK_DIR / 'artifacts'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [17, 42, 123, 2025, 2026]
PRIMARY_CAPACITY = 0.20
CAPACITIES = [0.10, 0.20, 0.30]
N_BOOTSTRAP = 1000
print('Work directory:', WORK_DIR)
print('Python:', platform.python_version(), '| pandas:', pd.__version__, '| scikit-learn:', __import__('sklearn').__version__)
print('Artifacts will be written to:', ARTIFACT_DIR)

In [ ]:
# Official CDC public-use files. Optional files are logged if not available in a cycle.
CYCLE_CONFIGS = {
    '2017_mar2020': {'label': '2017–March 2020 pre-pandemic', 'base_url': 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/', 'prefix': 'P_', 'suffix': ''},
    '2021_2023': {'label': 'August 2021–August 2023', 'base_url': 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/', 'prefix': '', 'suffix': '_L'},
}
FILE_STEMS = ('DEMO', 'DIQ', 'GHB', 'BMX', 'BPXO', 'BPQ', 'SMQ', 'PAQ', 'MCQ', 'HDL')
ESSENTIAL_STEMS = {'DEMO', 'DIQ', 'GHB', 'BMX'}
FILE_VARIABLES = {
    'DEMO': ['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','DMDEDUC2','INDFMPIR','WTMECPRP','WTMEC2YR','SDMVSTRA','SDMVPSU'],
    'DIQ': ['SEQN','DIQ010','DIQ160'],
    'GHB': ['SEQN','LBXGH'],
    'BMX': ['SEQN','BMXBMI','BMXWAIST'],
    'BPXO': ['SEQN','BPXOSY1','BPXOSY2','BPXOSY3','BPXOSY4','BPXODI1','BPXODI2','BPXODI3','BPXODI4'],
    'BPQ': ['SEQN','BPQ020','BPQ040','BPQ040A'],
    'SMQ': ['SEQN','SMQ020'],
    'PAQ': ['SEQN','PAQ605','PAQ610','PAQ615','PAQ620','PAQ625','PAQ630','PAQ635','PAQ640','PAQ645','PAQ650','PAQ655','PAQ660','PAD680','PAD790Q','PAD790U','PAD800','PAD810Q','PAD810U','PAD820']
    ,'HDL': ['SEQN','LBDHDD'],
    'MCQ': ['SEQN','MCQ160B','MCQ160C','MCQ160D','MCQ160E','MCQ160F'],
}

def read_xpt(url: str, cache_path: Path, timeout: int = 90) -> pd.DataFrame:
    if cache_path.exists():
        return pd.read_sas(cache_path, format='xport', encoding='utf-8')
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_bytes(response.content)
    return pd.read_sas(BytesIO(response.content), format='xport', encoding='utf-8')

def load_cycle(cycle: str):
    cfg = CYCLE_CONFIGS[cycle]
    merged, report = None, {'cycle': cycle, 'label': cfg['label'], 'loaded_files': [], 'skipped_files': [], 'missing_variables': {}}
    for stem in FILE_STEMS:
        filename = f"{cfg['prefix']}{stem}{cfg['suffix']}.xpt"
        url = cfg['base_url'] + filename
        try:
            data = read_xpt(url, CACHE_DIR / cycle / filename)
        except Exception as exc:
            if stem in ESSENTIAL_STEMS:
                raise RuntimeError(f'Could not load essential CDC file {url}: {exc}') from exc
            report['skipped_files'].append({'file': filename, 'reason': str(exc)})
            print(f'Optional source skipped: {filename} ({exc})')
            continue
        wanted = FILE_VARIABLES[stem]
        report['missing_variables'][stem] = [c for c in wanted if c not in data.columns]
        data = data[[c for c in wanted if c in data.columns]].copy()
        if 'SEQN' not in data.columns:
            if stem in ESSENTIAL_STEMS:
                raise RuntimeError(f'Essential file {filename} does not contain SEQN')
            report['skipped_files'].append({'file': filename, 'reason': 'SEQN missing'})
            continue
        if merged is None:
            merged = data
        else:
            duplicate = (set(merged.columns) & set(data.columns)) - {'SEQN'}
            if duplicate:
                data = data.drop(columns=list(duplicate))
            merged = merged.merge(data, on='SEQN', how='left', validate='one_to_one')
        report['loaded_files'].append(filename)
    if merged is None:
        raise RuntimeError(f'No data loaded for cycle {cycle}')
    merged['cycle'] = cycle
    report['rows_after_merge'] = int(len(merged))
    return merged, report

def numeric(frame, col):
    if col not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype='float64')
    return pd.to_numeric(frame[col], errors='coerce')

def clean_response(series, invalid=None):
    x = pd.to_numeric(series, errors='coerce').astype('float64')
    for bad in (invalid if invalid is not None else {7,9,77,99,777,999,7777,9999}):
        x = x.mask(x == bad)
    x = x.mask(x.abs() < 1e-300)
    return x

def mean_available(frame, cols):
    usable = [c for c in cols if c in frame.columns]
    if not usable:
        return pd.Series(np.nan, index=frame.index, dtype='float64')
    return frame[usable].apply(pd.to_numeric, errors='coerce').mean(axis=1, skipna=True)

def canonicalize(raw, cycle):
    out = pd.DataFrame(index=raw.index)
    out['SEQN'] = raw['SEQN']
    out['cycle'] = cycle
    out['age_years'] = numeric(raw, 'RIDAGEYR')
    out['sex_code'] = clean_response(raw['RIAGENDR'] if 'RIAGENDR' in raw else pd.Series(np.nan,index=raw.index), invalid=set())
    out['race_ethnicity_code'] = clean_response(raw['RIDRETH3'] if 'RIDRETH3' in raw else pd.Series(np.nan,index=raw.index), invalid=set())
    out['education_code'] = clean_response(raw['DMDEDUC2'] if 'DMDEDUC2' in raw else pd.Series(np.nan,index=raw.index))
    out['income_poverty_ratio'] = numeric(raw, 'INDFMPIR')
    out['bmi'] = numeric(raw, 'BMXBMI')
    out['waist_cm'] = numeric(raw, 'BMXWAIST')
    out['systolic_bp_mean'] = mean_available(raw, [f'BPXOSY{i}' for i in range(1,5)])
    out['diastolic_bp_mean'] = mean_available(raw, [f'BPXODI{i}' for i in range(1,5)])
    mapping = {
        'BPQ020':'hypertension_history','SMQ020':'ever_smoked_100_cigs',
        'MCQ160B':'heart_failure_history','MCQ160C':'coronary_heart_disease_history','MCQ160D':'angina_history',
        'MCQ160E':'heart_attack_history','MCQ160F':'stroke_history'
    }
    for source, target in mapping.items():
        out[target] = clean_response(raw[source] if source in raw else pd.Series(np.nan,index=raw.index))
    treatment_source = 'BPQ040A' if 'BPQ040A' in raw else 'BPQ040'
    out['hypertension_treatment'] = clean_response(raw[treatment_source] if treatment_source in raw else pd.Series(np.nan,index=raw.index))
    def mins(days_col, duration_col, yes_no_col):
        yes_no = clean_response(raw[yes_no_col] if yes_no_col in raw else pd.Series(np.nan,index=raw.index))
        days = clean_response(raw[days_col] if days_col in raw else pd.Series(np.nan,index=raw.index))
        duration = clean_response(raw[duration_col] if duration_col in raw else pd.Series(np.nan,index=raw.index))
        result = days * duration
        result.loc[yes_no.eq(2)] = 0.0
        return result
    if 'PAQ635' in raw.columns:
        out['leisure_activity_minutes'] = pd.concat([mins('PAQ640','PAQ645','PAQ635'), mins('PAQ655','PAQ660','PAQ650')],axis=1).sum(axis=1,min_count=1)
    elif 'PAD790Q' in raw.columns:
        def _temporal_weekly(qcol, ucol, mcol):
            q = clean_response(raw[qcol] if qcol in raw else pd.Series(np.nan,index=raw.index))
            u = (raw[ucol] if ucol in raw else pd.Series('',index=raw.index)).astype(str).str.strip().str.upper()
            m = clean_response(raw[mcol] if mcol in raw else pd.Series(np.nan,index=raw.index))
            factor = u.map({'D':7.0,'W':1.0,'M':52.0/12.0,'Y':1.0/52.0})
            return (q * m * factor)
        parts = [_temporal_weekly('PAD790Q','PAD790U','PAD800'), _temporal_weekly('PAD810Q','PAD810U','PAD820')]
        out['leisure_activity_minutes'] = pd.concat(parts,axis=1).sum(axis=1,min_count=1)
    else:
        out['leisure_activity_minutes'] = pd.Series(np.nan,index=raw.index,dtype='float64')
    out['sedentary_minutes'] = clean_response(raw['PAD680'] if 'PAD680' in raw else pd.Series(np.nan,index=raw.index))
    out['hdl_cholesterol'] = numeric(raw,'LBDHDD')
    out['reported_diabetes'] = clean_response(raw['DIQ010'] if 'DIQ010' in raw else pd.Series(np.nan,index=raw.index), invalid=set())
    out['reported_prediabetes'] = clean_response(raw['DIQ160'] if 'DIQ160' in raw else pd.Series(np.nan,index=raw.index), invalid=set())
    out['hba1c_percent'] = numeric(raw, 'LBXGH')
    for weight_col in ('WTMECPRP','WTMEC2YR','SDMVSTRA','SDMVPSU'):
        if weight_col in raw.columns:
            out[weight_col.lower()] = numeric(raw, weight_col)
    return out

def make_cohort(frame):
    x = frame.loc[(frame['age_years'] >= 18) & (frame['reported_diabetes'] == 2) & (frame['reported_prediabetes'] == 2) & frame['hba1c_percent'].notna()].copy()
    x['target'] = (x['hba1c_percent'] >= 5.7).astype('int8')
    x['target_diabetes_range'] = (x['hba1c_percent'] >= 6.5).astype('int8')
    return x.reset_index(drop=True)

historical_raw, historical_report = load_cycle('2017_mar2020')
temporal_raw, temporal_report = load_cycle('2021_2023')
historical = make_cohort(canonicalize(historical_raw, '2017_mar2020'))
temporal_test = make_cohort(canonicalize(temporal_raw, '2021_2023'))
del historical_raw, temporal_raw

print('Historical eligible cohort:', len(historical), '| positives:', int(historical['target'].sum()), '| prevalence:', round(float(historical['target'].mean()), 4))
print('Temporal test eligible cohort:', len(temporal_test), '| positives:', int(temporal_test['target'].sum()), '| prevalence:', round(float(temporal_test['target'].mean()), 4))
print('Files loaded historical:', historical_report['loaded_files'])
print('Files loaded temporal:', temporal_report['loaded_files'])
if len(historical) < 200 or len(temporal_test) < 100:
    warnings.warn('Cohort is smaller than expected. Inspect codebooks, source availability and exclusions before interpreting metrics.')
if historical['target'].nunique() < 2 or temporal_test['target'].nunique() < 2:
    raise ValueError('Both outcome classes are required in historical and temporal cohorts; inspect the source data and cohort rules.')


In [ ]:
ONTOLOGY_TTL_EMBEDDED = '@prefix : <https://w3id.org/clinontoscreen/ontology#> .\n@prefix owl: <http://www.w3.org/2002/07/owl#> .\n@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n@prefix dcterms: <http://purl.org/dc/terms/> .\n@prefix loinc: <https://loinc.org/> .\n\n<https://w3id.org/clinontoscreen/ontology> a owl:Ontology ;\n    dcterms:title "ClinOntoScreen diabetes screening knowledge model"@en ;\n    dcterms:description "A bounded, auditable representation of selected clinical screening criteria for research on NHANES. This is not a complete clinical guideline or a diagnostic system."@en ;\n    owl:versionInfo "0.1.1" ;\n    dcterms:created "2026-10-29"^^xsd:date ;\n    dcterms:source <https://diabetesjournals.org/care/article/49/Supplement_1/S27/163926/2-Diagnosis-and-Classification-of-Diabetes> .\n\n:Person a owl:Class ; rdfs:label "Person"@en .\n:ClinicalMeasurement a owl:Class ; rdfs:label "Clinical measurement"@en .\n:RiskFactor a owl:Class ; rdfs:label "Risk factor"@en .\n:ScreeningCriterion a owl:Class ; rdfs:label "Screening criterion"@en .\n:ScreeningRule a owl:Class ; rdfs:label "Screening rule"@en .\n:ScreeningAssessment a owl:Class ; rdfs:label "Screening assessment"@en .\n:OutcomeDefinition a owl:Class ; rdfs:label "Research outcome definition"@en ;\n    rdfs:comment "Outcome definitions are used only for evaluation and must not become predictors or screening rules."@en .\n:DataVariable a owl:Class ; rdfs:label "Dataset variable"@en .\n:GuidelineSource a owl:Class ; rdfs:label "Guideline source"@en .\n\n:hasMeasurement a owl:ObjectProperty ; rdfs:domain :Person ; rdfs:range :ClinicalMeasurement .\n:hasRiskFactor a owl:ObjectProperty ; rdfs:domain :Person ; rdfs:range :RiskFactor .\n:usesCriterion a owl:ObjectProperty ; rdfs:domain :ScreeningRule ; rdfs:range :ScreeningCriterion .\n:usesVariable a owl:ObjectProperty ; rdfs:domain :ScreeningRule ; rdfs:range :DataVariable .\n:derivedFrom a owl:ObjectProperty ; rdfs:domain :DataVariable ; rdfs:range :DataVariable .\n:hasSource a owl:ObjectProperty ; rdfs:range :GuidelineSource .\n:producesAssessment a owl:ObjectProperty ; rdfs:domain :ScreeningRule ; rdfs:range :ScreeningAssessment .\n:meetsCriterion a owl:ObjectProperty ; rdfs:domain :ScreeningAssessment ; rdfs:range :ScreeningCriterion .\n\n:ageYears a owl:DatatypeProperty ; rdfs:domain :Person ; rdfs:range xsd:integer .\n:bmi a owl:DatatypeProperty ; rdfs:domain :Person ; rdfs:range xsd:decimal .\n:systolicBloodPressure a owl:DatatypeProperty ; rdfs:domain :Person ; rdfs:range xsd:decimal .\n:diastolicBloodPressure a owl:DatatypeProperty ; rdfs:domain :Person ; rdfs:range xsd:decimal .\n:ruleExpression a owl:DatatypeProperty ; rdfs:domain :ScreeningRule ; rdfs:range xsd:string .\n:thresholdValue a owl:DatatypeProperty ; rdfs:domain :ScreeningCriterion ; rdfs:range xsd:decimal .\n:thresholdUnit a owl:DatatypeProperty ; rdfs:domain :ScreeningCriterion ; rdfs:range xsd:string .\n:requiresRiskFactorCount a owl:DatatypeProperty ; rdfs:domain :ScreeningRule ; rdfs:range xsd:integer .\n:systolicThreshold a owl:DatatypeProperty ; rdfs:domain :ScreeningCriterion ; rdfs:range xsd:decimal .\n:diastolicThreshold a owl:DatatypeProperty ; rdfs:domain :ScreeningCriterion ; rdfs:range xsd:decimal .\n:categoryCodes a owl:DatatypeProperty ; rdfs:domain :ScreeningCriterion ; rdfs:range xsd:string .\n:implementationNote a owl:DatatypeProperty ; rdfs:domain :ScreeningRule ; rdfs:range xsd:string .\n:ruleOrder a owl:DatatypeProperty ; rdfs:domain :ScreeningRule ; rdfs:range xsd:integer .\n:nhanesVariable a owl:DatatypeProperty ; rdfs:domain :DataVariable ; rdfs:range xsd:string .\n:sourceURL a owl:DatatypeProperty ; rdfs:domain :GuidelineSource ; rdfs:range xsd:anyURI .\n\n:ADAStandards2026 a :GuidelineSource ;\n    rdfs:label "American Diabetes Association Standards of Care in Diabetes—2026"@en ;\n    :sourceURL "https://diabetesjournals.org/care/article/49/Supplement_1/S27/163926/2-Diagnosis-and-Classification-of-Diabetes"^^xsd:anyURI ;\n    rdfs:comment "Selected screening recommendations are operationalized only where the required NHANES variables are available. Refer to the source document for the full recommendation."@en .\n\n:Age35Criterion a :ScreeningCriterion ;\n    rdfs:label "Adult age-based screening criterion"@en ;\n    rdfs:comment "ADA guidance recommends screening adults beginning at age 35. This project encodes this as a screening eligibility criterion, not as a diagnosis."@en ;\n    :thresholdValue "35"^^xsd:decimal ; :thresholdUnit "years" .\n\n:BMIOverweightGeneralCriterion a :ScreeningCriterion ;\n    rdfs:label "BMI threshold for overweight in general screening criteria"@en ;\n    :thresholdValue "25"^^xsd:decimal ; :thresholdUnit "kg/m2" ;\n    rdfs:comment "The selected ADA screening recommendation uses BMI >=25 kg/m2 for adults with overweight or obesity, subject to additional risk criteria when younger than the age-based screening threshold."@en .\n\n:BMIOverweightAsianCriterion a :ScreeningCriterion ;\n    rdfs:label "BMI threshold used for Asian ancestry in screening criteria"@en ;\n    :thresholdValue "23"^^xsd:decimal ; :thresholdUnit "kg/m2" ;\n    rdfs:comment "The selected ADA screening recommendation uses BMI >=23 kg/m2 for individuals of Asian ancestry. NHANES categories are an imperfect proxy for ancestry; this is a documented limitation and must be ablated in evaluation."@en .\n\n:HypertensionRiskCriterion a :ScreeningCriterion ;\n    rdfs:label "Hypertension risk factor"@en ;\n    :systolicThreshold "130"^^xsd:decimal ;\n    :diastolicThreshold "80"^^xsd:decimal ;\n    rdfs:comment "Operationalized from measured blood pressure, self-reported hypertension history, or prescribed hypertension treatment when available in NHANES."@en .\n\n:CVDHistoryRiskCriterion a :ScreeningCriterion ;\n    rdfs:label "Cardiovascular disease history risk factor"@en ;\n    rdfs:comment "Operationalized from selected self-reported cardiovascular history items. It is not a comprehensive cardiovascular history."@en .\n\n:PhysicalInactivityRiskCriterion a :ScreeningCriterion ;\n    rdfs:label "Physical inactivity risk factor"@en ;\n    rdfs:comment "Operationalized from selected NHANES work and recreational activity items when sufficient responses are available."@en .\n\n:HigherRiskEthnicityCriterion a :ScreeningCriterion ;\n    rdfs:label "Higher-risk race, ethnicity, or ancestry factor in screening guidance"@en ;\n    :categoryCodes "1,2,4,6" ;\n    rdfs:comment "Encodes only an approximate grouping supported by available NHANES categories. This is included as a guideline criterion for a sensitivity analysis, not interpreted as biological causation. The primary paper must discuss fairness and show an ablation without this feature."@en .\n\n:AgeBasedScreeningRule a :ScreeningRule ;\n    rdfs:label "Age-based screening rule"@en ;\n    :ruleOrder 1 ;\n    :ruleExpression "age_years >= 35" ;\n    :usesCriterion :Age35Criterion ;\n    :usesVariable :AgeVariable ;\n    :hasSource :ADAStandards2026 ;\n    :implementationNote "If age is known and >=35, mark screening eligibility. This is not a diagnosis and does not imply confirmed disease."@en .\n\n:YoungerAdultRiskBasedRule a :ScreeningRule ;\n    rdfs:label "Younger-adult BMI plus additional risk-factor rule"@en ;\n    :ruleOrder 2 ;\n    :ruleExpression "age_years < 35 AND bmi_meets_ancestry_threshold AND at_least_one_observed_additional_risk_factor" ;\n    :requiresRiskFactorCount 1 ;\n    :usesCriterion :BMIOverweightGeneralCriterion, :BMIOverweightAsianCriterion, :HypertensionRiskCriterion, :CVDHistoryRiskCriterion, :PhysicalInactivityRiskCriterion, :HigherRiskEthnicityCriterion ;\n    :usesVariable :AgeVariable, :BMIVariable, :RaceEthnicityVariable, :SystolicBPVariable, :DiastolicBPVariable, :HypertensionHistoryVariable, :HypertensionTreatmentVariable, :CVDHistoryVariable, :PhysicalActivityVariable ;\n    :hasSource :ADAStandards2026 ;\n    :implementationNote "This is a bounded operationalization of selected screening factors. Missing criteria are marked unknown; they are not treated as absent. The complete guideline contains other factors not represented here."@en .\n\n:AgeVariable a :DataVariable ; rdfs:label "Age in years"@en ; :nhanesVariable "RIDAGEYR" .\n:SexVariable a :DataVariable ; rdfs:label "NHANES sex/gender field (as coded in source)"@en ; :nhanesVariable "RIAGENDR" .\n:RaceEthnicityVariable a :DataVariable ; rdfs:label "NHANES race/Hispanic origin including non-Hispanic Asian category"@en ; :nhanesVariable "RIDRETH3" .\n:BMIVariable a :DataVariable ; rdfs:label "Body mass index"@en ; :nhanesVariable "BMXBMI" .\n:WaistVariable a :DataVariable ; rdfs:label "Waist circumference"@en ; :nhanesVariable "BMXWAIST" .\n:SystolicBPVariable a :DataVariable ; rdfs:label "Oscillometric systolic blood pressure reading"@en ; :nhanesVariable "BPXOSY1-BPXOSY4" .\n:DiastolicBPVariable a :DataVariable ; rdfs:label "Oscillometric diastolic blood pressure reading"@en ; :nhanesVariable "BPXODI1-BPXODI4" .\n:HypertensionHistoryVariable a :DataVariable ; rdfs:label "Ever told by a health professional that blood pressure was high"@en ; :nhanesVariable "BPQ020" .\n:HypertensionTreatmentVariable a :DataVariable ; rdfs:label "Currently taking prescribed medicine for hypertension"@en ; :nhanesVariable "BPQ040" .\n:CVDHistoryVariable a :DataVariable ; rdfs:label "Selected cardiovascular history questions"@en ; :nhanesVariable "MCQ160B-MCQ160F" .\n:PhysicalActivityVariable a :DataVariable ; rdfs:label "Selected physical activity questions"@en ; :nhanesVariable "PAQ605, PAQ620, PAQ650, PAQ665" .\n:SmokingVariable a :DataVariable ; rdfs:label "Ever smoked at least 100 cigarettes"@en ; :nhanesVariable "SMQ020" .\n:EducationVariable a :DataVariable ; rdfs:label "Education level for adults"@en ; :nhanesVariable "DMDEDUC2" .\n:IncomeVariable a :DataVariable ; rdfs:label "Family income-to-poverty ratio"@en ; :nhanesVariable "INDFMPIR" .\n\n:HbA1cTargetDefinition a :OutcomeDefinition ;\n    rdfs:label "Research outcome: HbA1c-defined screen-positive dysglycemia"@en ;\n    rdfs:comment "Among adults with valid HbA1c who report neither prior diabetes nor prior prediabetes, target=1 when LBXGH >=5.7%. This is a research label based on one survey measurement, not a confirmed clinical diagnosis."@en ;\n    :thresholdValue "5.7"^^xsd:decimal ; :thresholdUnit "percent HbA1c" ;\n    :usesVariable :HbA1cVariable ;\n    :hasSource :ADAStandards2026 .\n\n:DiabetesRangeSecondaryOutcome a :OutcomeDefinition ;\n    rdfs:label "Secondary outcome: single HbA1c measurement in diabetes range"@en ;\n    rdfs:comment "Among the same eligible cohort, target=1 when LBXGH >=6.5%. A single measurement is not by itself a confirmed diagnosis."@en ;\n    :thresholdValue "6.5"^^xsd:decimal ; :thresholdUnit "percent HbA1c" ;\n    :usesVariable :HbA1cVariable ;\n    :hasSource :ADAStandards2026 .\n\n:HbA1cVariable a :DataVariable ;\n    rdfs:label "NHANES glycohemoglobin (HbA1c) laboratory measurement"@en ;\n    :nhanesVariable "LBXGH" ;\n    rdfs:comment "Outcome-only variable. Strictly prohibited from the ML feature matrix."@en ;\n    :derivedFrom loinc:4548-4 .\n\nloinc:4548-4 rdfs:label "Hemoglobin A1c/Hemoglobin.total in Blood"@en .\n'
ontology_candidates = [Path.cwd() / 'ontology' / 'diabetes_screening.ttl', Path.cwd().parent / 'ontology' / 'diabetes_screening.ttl']
ontology_path = next((p for p in ontology_candidates if p.exists()), None)
ontology_graph = Graph()
if ontology_path:
    ontology_graph.parse(ontology_path, format='turtle')
    print('Loaded repository ontology:', ontology_path)
else:
    ontology_graph.parse(data=ONTOLOGY_TTL_EMBEDDED, format='turtle')
    ontology_path = ARTIFACT_DIR / 'diabetes_screening.ttl'
    ontology_path.write_text(ONTOLOGY_TTL_EMBEDDED, encoding='utf-8')
    print('Loaded embedded ontology and wrote a copy to:', ontology_path)

EX_NS = 'https://w3id.org/clinontoscreen/ontology#'
def ontology_parameter(subject_name, predicate_name, default):
    from rdflib import URIRef
    subject = URIRef(EX_NS + subject_name)
    predicate = URIRef(EX_NS + predicate_name)
    values = list(ontology_graph.objects(subject, predicate))
    return float(values[0]) if values else float(default)
AGE_THRESHOLD = ontology_parameter('Age35Criterion', 'thresholdValue', 35)
BMI_GENERAL = ontology_parameter('BMIOverweightGeneralCriterion', 'thresholdValue', 25)
BMI_ASIAN = ontology_parameter('BMIOverweightAsianCriterion', 'thresholdValue', 23)
SBP_THRESHOLD = ontology_parameter('HypertensionRiskCriterion', 'systolicThreshold', 130)
DBP_THRESHOLD = ontology_parameter('HypertensionRiskCriterion', 'diastolicThreshold', 80)
from rdflib import URIRef
_codes = list(ontology_graph.objects(URIRef(EX_NS+'HigherRiskEthnicityCriterion'), URIRef(EX_NS+'categoryCodes')))
HIGHER_RISK_ETHNICITY_CODES = [int(x.strip()) for x in (str(_codes[0]) if _codes else '1,2,4,6').split(',')]
print('Ontology triples:', len(ontology_graph), '| rules:', AGE_THRESHOLD, BMI_GENERAL, BMI_ASIAN, SBP_THRESHOLD, DBP_THRESHOLD)

RAW_NUMERIC = ['age_years','income_poverty_ratio','bmi','waist_cm','systolic_bp_mean','diastolic_bp_mean','hdl_cholesterol','leisure_activity_minutes','sedentary_minutes']
RAW_CATEGORICAL = ['sex_code','race_ethnicity_code','education_code','hypertension_history','ever_smoked_100_cigs','heart_failure_history','coronary_heart_disease_history','angina_history','heart_attack_history','stroke_history']
RAW_FEATURES = RAW_NUMERIC + RAW_CATEGORICAL

# Data-QC gate: no selected model feature may be entirely missing in either cycle.
combined_qc = pd.concat([historical, temporal_test], ignore_index=True)
for cycle_name, cycle_frame in combined_qc.groupby('cycle'):
    missingness = cycle_frame[RAW_FEATURES].isna().mean().sort_values(ascending=False)
    print('Feature missingness', cycle_name); display(missingness.to_frame('fraction_missing'))
    assert not (missingness >= 1.0).any(), f'100% missing selected feature(s) in {cycle_name}: {missingness[missingness >= 1.0].index.tolist()}'
ONTOLOGY_FEATURES = ['onto_age_based_screening','onto_bmi_threshold_met','onto_hypertension_risk','onto_cvd_history','onto_physical_inactivity','onto_higher_risk_ethnicity','onto_additional_risk_factor_count','onto_screening_eligible','onto_criteria_observed_count','onto_criteria_missing_count','onto_rule_priority_score']
MANUAL_FEATURES = [c.replace('onto_', 'manual_') for c in ONTOLOGY_FEATURES]
FORBIDDEN_FEATURE_TOKENS = ('hba1c','glucose','reported_diabetes','reported_prediabetes','target','diabetes_medication')
assert not any(any(token in c.lower() for token in FORBIDDEN_FEATURE_TOKENS) for c in RAW_FEATURES + ONTOLOGY_FEATURES)
assert 'SEQN' not in RAW_FEATURES + ONTOLOGY_FEATURES

def yes_no_flag(frame, col):
    vals = pd.to_numeric(frame[col], errors='coerce') if col in frame else pd.Series(np.nan,index=frame.index,dtype='float64')
    return vals.map({1.0:1.0,2.0:0.0}).astype('float64')

def any_flag(flags, index):
    values = pd.concat(flags, axis=1)
    result = pd.Series(np.nan,index=index,dtype='float64')
    any_yes = values.eq(1).any(axis=1); any_known = values.notna().any(axis=1)
    result.loc[any_yes] = 1.0; result.loc[~any_yes & any_known] = 0.0
    return result

def derive_ontology_features(frame, include_ethnicity=True):
    out = frame.copy()
    age = pd.to_numeric(out.get('age_years',pd.Series(np.nan,index=out.index)),errors='coerce')
    bmi = pd.to_numeric(out.get('bmi',pd.Series(np.nan,index=out.index)),errors='coerce')
    race = pd.to_numeric(out.get('race_ethnicity_code',pd.Series(np.nan,index=out.index)),errors='coerce')
    asian = race.eq(6); cutoff = pd.Series(BMI_GENERAL,index=out.index,dtype='float64'); cutoff.loc[asian] = BMI_ASIAN
    bmi_flag = pd.Series(np.nan,index=out.index,dtype='float64'); known_bmi = bmi.notna() & race.notna(); bmi_flag.loc[known_bmi] = (bmi.loc[known_bmi] >= cutoff.loc[known_bmi]).astype(float)
    age_flag = pd.Series(np.nan,index=out.index,dtype='float64'); age_flag.loc[age.notna()] = (age.loc[age.notna()] >= AGE_THRESHOLD).astype(float)
    sbp = pd.to_numeric(out.get('systolic_bp_mean',pd.Series(np.nan,index=out.index)),errors='coerce')
    dbp = pd.to_numeric(out.get('diastolic_bp_mean',pd.Series(np.nan,index=out.index)),errors='coerce')
    measured_bp = pd.Series(np.nan,index=out.index,dtype='float64'); bp_known = sbp.notna() | dbp.notna()
    measured_bp.loc[bp_known] = (((sbp >= SBP_THRESHOLD) | (dbp >= DBP_THRESHOLD)).loc[bp_known]).astype(float)
    hypertension = any_flag([measured_bp,yes_no_flag(out,'hypertension_history'),yes_no_flag(out,'hypertension_treatment')],out.index)
    cvd = any_flag([yes_no_flag(out,c) for c in ['heart_failure_history','coronary_heart_disease_history','angina_history','heart_attack_history','stroke_history']],out.index)
    leisure = pd.to_numeric(out.get('leisure_activity_minutes',pd.Series(np.nan,index=out.index)),errors='coerce')
    sedentary = pd.to_numeric(out.get('sedentary_minutes',pd.Series(np.nan,index=out.index)),errors='coerce')
    inactivity = pd.Series(np.nan,index=out.index,dtype='float64'); observed_activity = leisure.notna() | sedentary.notna()
    inactivity.loc[observed_activity] = ((leisure.fillna(0) <= 0) & (sedentary.fillna(0) >= 480)).loc[observed_activity].astype(float)
    ethnicity = pd.Series(np.nan,index=out.index,dtype='float64')
    if include_ethnicity:
        ethnicity.loc[race.notna()] = race.loc[race.notna()].isin(HIGHER_RISK_ETHNICITY_CODES).astype(float)
    else:
        ethnicity = pd.Series(0.0,index=out.index,dtype='float64')
    risk_columns = [hypertension,cvd,inactivity]
    if include_ethnicity: risk_columns.append(ethnicity)
    risks = pd.concat(risk_columns,axis=1); risk_count = risks.fillna(0).sum(axis=1).astype(float); risk_count.loc[risks.notna().sum(axis=1)==0] = np.nan
    eligible = pd.Series(np.nan,index=out.index,dtype='float64'); eligible.loc[age >= AGE_THRESHOLD] = 1.0
    younger = age.notna() & (age < AGE_THRESHOLD) & bmi_flag.notna(); eligible.loc[younger & bmi_flag.eq(0)] = 0.0
    eligible.loc[younger & bmi_flag.eq(1) & (risk_count >= 1)] = 1.0; eligible.loc[younger & bmi_flag.eq(1) & risk_count.eq(0)] = 0.0
    criteria_columns = [age_flag,bmi_flag,hypertension,cvd,inactivity]
    if include_ethnicity: criteria_columns.append(ethnicity)
    criteria = pd.concat(criteria_columns,axis=1); observed = criteria.notna().sum(axis=1).astype(float); missing = criteria.isna().sum(axis=1).astype(float)
    priority = criteria.fillna(0).sum(axis=1).astype(float) / observed.replace(0,np.nan)
    vals = [age_flag,bmi_flag,hypertension,cvd,inactivity,ethnicity,risk_count,eligible,observed,missing,priority.fillna(0.0)]
    for name,val in zip(ONTOLOGY_FEATURES,vals): out[name]=val
    return out

def derive_manual_control(frame):
    out = frame.copy();
    if 'hypertension_treatment' not in out.columns: out['hypertension_treatment'] = 2.0;
    age=pd.to_numeric(out['age_years'],errors='coerce'); bmi=pd.to_numeric(out['bmi'],errors='coerce'); race=pd.to_numeric(out['race_ethnicity_code'],errors='coerce')
    threshold=np.where(race.eq(6),BMI_ASIAN,BMI_GENERAL); bmi_flag=pd.Series(np.where(bmi.notna()&race.notna(),(bmi>=threshold).astype(float),np.nan),index=out.index)
    age_flag=pd.Series(np.where(age.notna(),(age>=AGE_THRESHOLD).astype(float),np.nan),index=out.index)
    sbp=pd.to_numeric(out['systolic_bp_mean'],errors='coerce'); dbp=pd.to_numeric(out['diastolic_bp_mean'],errors='coerce')
    measured=pd.Series(np.where(sbp.notna()|dbp.notna(),((sbp>=SBP_THRESHOLD)|(dbp>=DBP_THRESHOLD)).astype(float),np.nan),index=out.index)
    def yn(col): return pd.to_numeric(out[col],errors='coerce').map({1.0:1.0,2.0:0.0}).astype('float64')
    def or_flags(items):
        z=pd.concat(items,axis=1); a=z.eq(1).any(axis=1); known=z.notna().any(axis=1); r=pd.Series(np.nan,index=out.index,dtype='float64'); r.loc[a]=1.0; r.loc[~a&known]=0.0; return r
    hypertension=or_flags([measured,yn('hypertension_history'),yn('hypertension_treatment')])
    cvd=or_flags([yn(c) for c in ['heart_failure_history','coronary_heart_disease_history','angina_history','heart_attack_history','stroke_history']])
    leisure=pd.to_numeric(out.get('leisure_activity_minutes', pd.Series(np.nan,index=out.index)),errors='coerce'); sedentary=pd.to_numeric(out.get('sedentary_minutes', pd.Series(np.nan,index=out.index)),errors='coerce')
    inactive=pd.Series(np.nan,index=out.index,dtype='float64'); observed=leisure.notna()|sedentary.notna(); inactive.loc[observed]=((leisure.fillna(0)<=0)&(sedentary.fillna(0)>=480)).loc[observed].astype(float)
    ethnicity=pd.Series(np.nan,index=out.index,dtype='float64'); ethnicity.loc[race.notna()]=race.loc[race.notna()].isin(HIGHER_RISK_ETHNICITY_CODES).astype(float)
    risks=pd.concat([hypertension,cvd,inactive,ethnicity],axis=1); count=risks.fillna(0).sum(axis=1).astype(float); count.loc[risks.notna().sum(axis=1)==0]=np.nan
    eligible=pd.Series(np.nan,index=out.index,dtype='float64'); eligible.loc[age>=AGE_THRESHOLD]=1.; younger=age.notna()&(age<AGE_THRESHOLD)&bmi_flag.notna(); eligible.loc[younger&bmi_flag.eq(0)]=0.; eligible.loc[younger&bmi_flag.eq(1)&(count>=1)]=1.; eligible.loc[younger&bmi_flag.eq(1)&count.eq(0)]=0.
    criteria=pd.concat([age_flag,bmi_flag,hypertension,cvd,inactive,ethnicity],axis=1); observed=criteria.notna().sum(axis=1).astype(float); missing=criteria.isna().sum(axis=1).astype(float); priority=criteria.fillna(0).sum(axis=1).astype(float)/observed.replace(0,np.nan)
    vals=[age_flag,bmi_flag,hypertension,cvd,inactive,ethnicity,count,eligible,observed,missing,priority.fillna(0.)]
    for name,val in zip(MANUAL_FEATURES,vals): out[name]=val
    return out

rule_base='https://w3id.org/clinontoscreen/ontology#'
from rdflib import URIRef
for rn in ['AgeBasedScreeningRule','YoungerAdultRiskBasedRule']:
    r=URIRef(rule_base+rn)
    assert list(ontology_graph.objects(r,URIRef(rule_base+'hasSource'))), f'{rn} has no source'
    assert list(ontology_graph.objects(r,URIRef(rule_base+'ruleExpression'))), f'{rn} has no expression'
print('Ontology source and rule-expression checks passed.')


In [ ]:
# Data quality, cohort flow, missingness, and leakage contract.
for label, df in [('historical', historical), ('temporal_test', temporal_test)]:
    print(f"\n{label}: n={len(df):,}; positives={int(df['target'].sum()):,}; prevalence={df['target'].mean():.3f}; diabetes-range secondary positives={int(df['target_diabetes_range'].sum()):,}")
    display(df[RAW_FEATURES].isna().mean().sort_values(ascending=False).rename('missing_fraction').to_frame().head(12))

cohort_qc = {
    'historical_n': int(len(historical)), 'historical_positive_count': int(historical['target'].sum()),
    'historical_prevalence': float(historical['target'].mean()),
    'temporal_test_n': int(len(temporal_test)), 'temporal_test_positive_count': int(temporal_test['target'].sum()),
    'temporal_test_prevalence': float(temporal_test['target'].mean()),
    'historical_missing_fraction': historical[RAW_FEATURES].isna().mean().to_dict(),
    'temporal_missing_fraction': temporal_test[RAW_FEATURES].isna().mean().to_dict(),
    'historical_source_report': historical_report, 'temporal_source_report': temporal_report,
}
(ARTIFACT_DIR / 'data_qc.json').write_text(json.dumps(cohort_qc, indent=2, default=str), encoding='utf-8')

for col in RAW_FEATURES:
    assert col not in {'SEQN','target','target_diabetes_range','hba1c_percent','reported_diabetes','reported_prediabetes'}
assert not any(any(token in col.lower() for token in FORBIDDEN_FEATURE_TOKENS) for col in RAW_FEATURES)
assert 'SEQN' not in RAW_FEATURES
print('Leakage contract passed. The cohort labels come from HbA1c; target-source fields are not model features.')

# Visualize class balance and age distribution by period.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(['Historical','Temporal test'], [historical['target'].mean(), temporal_test['target'].mean()])
axes[0].set_title('Primary outcome prevalence'); axes[0].set_ylabel('Fraction positive'); axes[0].set_ylim(0,1)
axes[1].hist(historical['age_years'].dropna(), bins=20, alpha=0.6, label='Historical')
axes[1].hist(temporal_test['age_years'].dropna(), bins=20, alpha=0.6, label='Temporal test')
axes[1].set_title('Age distribution'); axes[1].set_xlabel('Age (NHANES top-codes 80+)'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Model pipelines and metrics. All transforms are fitted inside each pipeline.
RAW_FEATURES = RAW_NUMERIC + RAW_CATEGORICAL

# Data-QC gate: no selected model feature may be entirely missing in either cycle.
combined_qc = pd.concat([historical, temporal_test], ignore_index=True)
for cycle_name, cycle_frame in combined_qc.groupby('cycle'):
    missingness = cycle_frame[RAW_FEATURES].isna().mean().sort_values(ascending=False)
    print('Feature missingness', cycle_name); display(missingness.to_frame('fraction_missing'))
    assert not (missingness >= 1.0).any(), f'100% missing selected feature(s) in {cycle_name}: {missingness[missingness >= 1.0].index.tolist()}'

def one_hot_encoder():
    try: return OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError: return OneHotEncoder(handle_unknown='ignore', sparse=True)

def make_preprocessor(features):
    numeric_cols = [c for c in features if c in RAW_NUMERIC or c.startswith('onto_') or c.startswith('manual_')]
    categorical_cols = [c for c in features if c not in numeric_cols]
    steps=[]
    if numeric_cols:
        steps.append(('numeric', Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]), numeric_cols))
    if categorical_cols:
        steps.append(('categorical', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',one_hot_encoder())]), categorical_cols))
    return ColumnTransformer(steps, remainder='drop', verbose_feature_names_out=False)

def make_pipeline(model_name, features, seed=42, params=None):
    params = dict(params or {})
    if model_name == 'logistic':
        base = {'C':1.0, 'max_iter':3000, 'solver':'lbfgs', 'random_state':seed}; base.update(params)
        estimator = LogisticRegression(**base)
    elif model_name == 'xgboost':
        base = {'n_estimators':250,'max_depth':3,'learning_rate':0.05,'subsample':0.85,'colsample_bytree':0.85,'reg_lambda':1.0,'min_child_weight':3,'eval_metric':'logloss','n_jobs':2,'random_state':seed,'tree_method':'hist'}; base.update(params)
        estimator = XGBClassifier(**base)
    else:
        raise ValueError(model_name)
    return Pipeline([('preprocess',make_preprocessor(features)),('model',estimator)])

def predict_prob(model, X): return np.asarray(model.predict_proba(X)[:,1],dtype=float)

def metric_pack(y, p):
    y=np.asarray(y,dtype=int); p=np.asarray(p,dtype=float)
    return {'n':int(len(y)),'positive_count':int(y.sum()),'prevalence':float(y.mean()),
            'auroc':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan,
            'auprc':float(average_precision_score(y,p)) if len(np.unique(y))==2 else np.nan,
            'brier':float(brier_score_loss(y,p))}



def simple_flag_metrics(y, score, capacity):
    y=np.asarray(y,dtype=int); score=np.asarray(score,dtype=float)
    n_selected=int(np.ceil(len(y)*capacity)); order=np.argsort(-np.nan_to_num(score,nan=-np.inf),kind='mergesort')
    pred=np.zeros(len(y),dtype=int); pred[order[:n_selected]]=1
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); positives=int(y.sum())
    return {'capacity':float(capacity),'n':int(len(y)),'n_selected':n_selected,'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn),'accuracy':float((tp+tn)/len(y)),'precision':float(tp/(tp+fp)) if tp+fp else 0.0,'recall':float(tp/positives) if positives else np.nan,'sensitivity':float(tp/positives) if positives else np.nan,'specificity':float(tn/(tn+fp)) if tn+fp else np.nan,'f1':float(f1_score(y,pred,zero_division=0))}

def capacity_pack(y, score, capacity):
    y=np.asarray(y,dtype=int); score=np.asarray(score,dtype=float)
    n=max(1,int(np.ceil(len(y)*capacity)))
    idx=np.argsort(-np.nan_to_num(score,nan=-np.inf),kind='mergesort')[:n]
    found=int(y[idx].sum()); pos=int(y.sum())
    return {'capacity':float(capacity),'n_selected':int(n),'positive_total':pos,'positive_found':found,
            'sensitivity':float(found/pos) if pos else np.nan,'precision':float(found/n),'false_referrals':int(n-found)}

# Independent historical validation is used for model/policy selection. The later cycle remains locked.
hist_raw = historical[RAW_FEATURES].copy()
hist_onto = derive_ontology_features(hist_raw)[RAW_FEATURES + ONTOLOGY_FEATURES].copy()
hist_onto_no_ethnicity = derive_ontology_features(hist_raw, include_ethnicity=False)[RAW_FEATURES + ONTOLOGY_FEATURES].copy()
hist_manual = derive_manual_control(hist_raw)[RAW_FEATURES + MANUAL_FEATURES].copy()
# Exact equivalence of values between ontology-derived and independently coded control features is expected.
for of, mf in zip(ONTOLOGY_FEATURES, MANUAL_FEATURES):
    a = hist_onto[of].to_numpy(dtype=float); b = hist_manual[mf].to_numpy(dtype=float)
    assert np.allclose(a, b, equal_nan=True), f'Feature control mismatch: {of} vs {mf}'
print('Generic feature-engineering control matches the same rule values independently.')

train_idx, val_idx = train_test_split(np.arange(len(historical)), test_size=0.20, random_state=42, stratify=historical['target'])
y_all = historical['target'].astype(int).to_numpy()
y_train, y_val = y_all[train_idx], y_all[val_idx]
X_variants = {
    'logistic_raw': (hist_raw, RAW_FEATURES, 'logistic'),
    'xgboost_raw': (hist_raw, RAW_FEATURES, 'xgboost'),
    'logistic_ontology': (hist_onto, RAW_FEATURES+ONTOLOGY_FEATURES, 'logistic'),
    'xgboost_ontology': (hist_onto, RAW_FEATURES+ONTOLOGY_FEATURES, 'xgboost'),
    'xgboost_ontology_no_ethnicity': (hist_onto_no_ethnicity, RAW_FEATURES+ONTOLOGY_FEATURES, 'xgboost'),
    'xgboost_generic_control': (hist_manual, RAW_FEATURES+MANUAL_FEATURES, 'xgboost'),
}
param_grids = {
    'logistic': {'model__C':[0.1,1.0,10.0]},
    'xgboost': {'model__n_estimators':[150,300], 'model__max_depth':[2,3], 'model__learning_rate':[0.03,0.07]},
}
searches, best_params, validation_probabilities = {}, {}, {}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for name, (X_all_variant, features, family) in X_variants.items():
    X_train = X_all_variant.iloc[train_idx][features]
    X_val = X_all_variant.iloc[val_idx][features]
    search = GridSearchCV(make_pipeline(family,features,seed=42), param_grids[family], scoring='average_precision', cv=cv, n_jobs=1, refit=True, verbose=0, return_train_score=False)
    search.fit(X_train, y_train)
    searches[name] = search
    best_params[name] = {key.replace('model__',''): value for key,value in search.best_params_.items()}
    validation_probabilities[name] = predict_prob(search.best_estimator_, X_val)
    print(name, '| best CV AUPRC:', round(float(search.best_score_),4), '| params:', best_params[name])

val_onto = hist_onto.iloc[val_idx]
val_rule_score = val_onto['onto_rule_priority_score'].to_numpy(dtype=float)
alpha_candidates = [0.0,0.025,0.05,0.10,0.15,0.20,0.30]
alpha_results=[]
for alpha in alpha_candidates:
    result=capacity_pack(y_val, validation_probabilities['xgboost_raw'] + alpha*val_rule_score, PRIMARY_CAPACITY)
    alpha_results.append({'alpha':alpha,**result})
alpha_results=sorted(alpha_results,key=lambda r:(r['sensitivity'] if np.isfinite(r['sensitivity']) else -1,r['precision'] if np.isfinite(r['precision']) else -1,-r['alpha']),reverse=True)
HYBRID_ALPHA=float(alpha_results[0]['alpha'])
print('Historical validation selects hybrid alpha:', HYBRID_ALPHA)
display(pd.DataFrame(alpha_results))

# Prespecified fairness/sensitivity ablation: remove only the higher-risk ethnicity risk-factor flag.
# The ADA BMI threshold adaptation for Asian ancestry remains in this specific ablation and is disclosed.
val_onto_no_ethnicity = hist_onto_no_ethnicity.iloc[val_idx]
val_rule_score_no_ethnicity = val_onto_no_ethnicity['onto_rule_priority_score'].to_numpy(dtype=float)
alpha_results_no_ethnicity=[]
for alpha in alpha_candidates:
    result=capacity_pack(y_val, validation_probabilities['xgboost_raw'] + alpha*val_rule_score_no_ethnicity, PRIMARY_CAPACITY)
    alpha_results_no_ethnicity.append({'alpha':alpha,**result})
alpha_results_no_ethnicity=sorted(alpha_results_no_ethnicity,key=lambda r:(r['sensitivity'] if np.isfinite(r['sensitivity']) else -1,r['precision'] if np.isfinite(r['precision']) else -1,-r['alpha']),reverse=True)
HYBRID_ALPHA_NO_ETHNICITY=float(alpha_results_no_ethnicity[0]['alpha'])
print('Historical validation selects no-higher-risk-ethnicity alpha:', HYBRID_ALPHA_NO_ETHNICITY)
display(pd.DataFrame(alpha_results_no_ethnicity))


In [ ]:
# Final training: refit selected configurations on all historical data across five seeds.
# The 2021–2023 temporal test is not used for selecting hyperparameters or alpha.
models = {name: [] for name in X_variants}
test_raw = temporal_test[RAW_FEATURES].copy()
test_onto = derive_ontology_features(test_raw)[RAW_FEATURES + ONTOLOGY_FEATURES].copy()
test_onto_no_ethnicity = derive_ontology_features(test_raw, include_ethnicity=False)[RAW_FEATURES + ONTOLOGY_FEATURES].copy()
test_manual = derive_manual_control(test_raw)[RAW_FEATURES + MANUAL_FEATURES].copy()
y_test = temporal_test['target'].astype(int).to_numpy()
X_hist_variants = {
    'logistic_raw': (hist_raw, RAW_FEATURES, 'logistic'),
    'xgboost_raw': (hist_raw, RAW_FEATURES, 'xgboost'),
    'logistic_ontology': (hist_onto, RAW_FEATURES+ONTOLOGY_FEATURES, 'logistic'),
    'xgboost_ontology': (hist_onto, RAW_FEATURES+ONTOLOGY_FEATURES, 'xgboost'),
    'xgboost_ontology_no_ethnicity': (hist_onto_no_ethnicity, RAW_FEATURES+ONTOLOGY_FEATURES, 'xgboost'),
    'xgboost_generic_control': (hist_manual, RAW_FEATURES+MANUAL_FEATURES, 'xgboost'),
}
for seed in SEEDS:
    print('Training seed', seed)
    for name, (X_train_full, features, family) in X_hist_variants.items():
        params = best_params[name]
        pipeline = make_pipeline(family, features, seed=seed, params=params)
        pipeline.fit(X_train_full[features], y_all)
        models[name].append(pipeline)

X_test_variants = {
    'logistic_raw': test_raw[RAW_FEATURES],
    'xgboost_raw': test_raw[RAW_FEATURES],
    'logistic_ontology': test_onto[RAW_FEATURES+ONTOLOGY_FEATURES],
    'xgboost_ontology': test_onto[RAW_FEATURES+ONTOLOGY_FEATURES],
    'xgboost_ontology_no_ethnicity': test_onto_no_ethnicity[RAW_FEATURES+ONTOLOGY_FEATURES],
    'xgboost_generic_control': test_manual[RAW_FEATURES+MANUAL_FEATURES],
}
probabilities = {name: np.mean(np.vstack([predict_prob(m, X_test_variants[name]) for m in model_list]),axis=0) for name, model_list in models.items()}
metrics_by_model = {}
per_seed_metrics = {}
for name, model_list in models.items():
    seed_metrics=[metric_pack(y_test,predict_prob(model,X_test_variants[name])) for model in model_list]
    per_seed_metrics[name]=seed_metrics
    metrics_by_model[name]={}
    for metric in ['auroc','auprc','brier']:
        vals=np.asarray([m[metric] for m in seed_metrics],dtype=float)
        metrics_by_model[name][metric+'_mean']=float(np.nanmean(vals))
        metrics_by_model[name][metric+'_sd']=float(np.nanstd(vals,ddof=1))
    metrics_by_model[name]['ensemble'] = metric_pack(y_test, probabilities[name])
    metrics_by_model[name]['cv_best_params'] = best_params[name]

display(pd.DataFrame([{ 'model':k, **{m:v for m,v in val.items() if m.endswith('_mean') or m.endswith('_sd')} } for k,val in metrics_by_model.items()]))
print('Temporal test participants:',len(y_test),'| positive cases:',int(y_test.sum()),'| prevalence:',round(float(y_test.mean()),4))

In [ ]:
# Fixed-capacity results, paired bootstrap uncertainty, and model–guideline disagreement.
test_rule_score = test_onto['onto_rule_priority_score'].to_numpy(dtype=float)
test_rule_score_no_ethnicity = test_onto_no_ethnicity['onto_rule_priority_score'].to_numpy(dtype=float)
test_eligible = test_onto['onto_screening_eligible'].to_numpy(dtype=float)
hybrid_score = probabilities['xgboost_raw'] + HYBRID_ALPHA*test_rule_score
hybrid_score_no_ethnicity = probabilities['xgboost_raw'] + HYBRID_ALPHA_NO_ETHNICITY*test_rule_score_no_ethnicity
screening_budget_rows=[]
policy_scores={
    'XGBoost raw': probabilities['xgboost_raw'],
    'XGBoost + ontology features': probabilities['xgboost_ontology'],
    'XGBoost + ontology features (no higher-risk ethnicity flag)': probabilities['xgboost_ontology_no_ethnicity'],
    'XGBoost + ontology policy': hybrid_score,
    'XGBoost + ontology policy (no higher-risk ethnicity flag)': hybrid_score_no_ethnicity,
    'Ontology-only rule score': test_rule_score,
    'Ontology-only rule score (no higher-risk ethnicity flag)': test_rule_score_no_ethnicity,
    'XGBoost + generic equivalent features': probabilities['xgboost_generic_control'],
}
for capacity in CAPACITIES:
    for policy,score in policy_scores.items():
        screening_budget_rows.append({'policy':policy,**capacity_pack(y_test,score,capacity)})
screening_budget_df=pd.DataFrame(screening_budget_rows)
display(screening_budget_df)

# Paired bootstrap difference in sensitivity at 20% capacity. The same resampled participants are used for both policies.
def sensitivity_from_scores(y, score, capacity=PRIMARY_CAPACITY):
    return capacity_pack(y,score,capacity)['sensitivity']
def paired_capacity_bootstrap(y, base_score, hybrid, n_bootstrap=N_BOOTSTRAP, seed=2026):
    y=np.asarray(y,dtype=int); base_score=np.asarray(base_score,dtype=float); hybrid=np.asarray(hybrid,dtype=float)
    rng=np.random.default_rng(seed); diffs=[]; attempts=0
    while len(diffs)<n_bootstrap and attempts < n_bootstrap*3:
        attempts += 1
        idx=rng.integers(0,len(y),size=len(y))
        if y[idx].sum()==0: continue
        diff=sensitivity_from_scores(y[idx],hybrid[idx])-sensitivity_from_scores(y[idx],base_score[idx])
        if np.isfinite(diff): diffs.append(float(diff))
    if not diffs: return {'difference':float('nan'),'ci_low':float('nan'),'ci_high':float('nan'),'n_bootstrap':0}
    return {'difference':float(sensitivity_from_scores(y,hybrid)-sensitivity_from_scores(y,base_score)),
            'ci_low':float(np.quantile(diffs,.025)),'ci_high':float(np.quantile(diffs,.975)),'n_bootstrap':len(diffs)}
primary_effect=paired_capacity_bootstrap(y_test,probabilities['xgboost_raw'],hybrid_score)
base_at20=capacity_pack(y_test,probabilities['xgboost_raw'],PRIMARY_CAPACITY)
hybrid_at20=capacity_pack(y_test,hybrid_score,PRIMARY_CAPACITY)
primary_effect.update({'baseline_sensitivity':base_at20['sensitivity'],'hybrid_sensitivity':hybrid_at20['sensitivity'],
                       'baseline_positive_found':base_at20['positive_found'],'hybrid_positive_found':hybrid_at20['positive_found'],
                       'capacity':PRIMARY_CAPACITY})
print('Primary endpoint, hybrid minus ML-only sensitivity:',primary_effect)
no_eth_base_at20=capacity_pack(y_test,probabilities['xgboost_raw'],PRIMARY_CAPACITY)
no_eth_hybrid_at20=capacity_pack(y_test,hybrid_score_no_ethnicity,PRIMARY_CAPACITY)
primary_effect_no_ethnicity=paired_capacity_bootstrap(y_test,probabilities['xgboost_raw'],hybrid_score_no_ethnicity)
primary_effect_no_ethnicity.update({'baseline_sensitivity':no_eth_base_at20['sensitivity'],'hybrid_sensitivity':no_eth_hybrid_at20['sensitivity'],
                                    'baseline_positive_found':no_eth_base_at20['positive_found'],'hybrid_positive_found':no_eth_hybrid_at20['positive_found'],
                                    'capacity':PRIMARY_CAPACITY})
print('Primary endpoint after removing the higher-risk ethnicity rule flag:',primary_effect_no_ethnicity)

# Confidence-only review baseline vs disagreement group at a historical-validation threshold.
val_raw_p=validation_probabilities['xgboost_raw']
threshold_grid=np.linspace(0.05,0.95,91)
threshold_scores=[]
for threshold in threshold_grid:
    pred=(val_raw_p>=threshold).astype(int)
    threshold_scores.append((f1_score(y_val,pred,zero_division=0),-abs(threshold-.5),float(threshold)))
VALIDATION_THRESHOLD=max(threshold_scores)[2]
test_pred=(probabilities['xgboost_raw']>=VALIDATION_THRESHOLD).astype(int)
rule_known=np.isfinite(test_eligible)
guideline_positive=np.where(rule_known,test_eligible==1,False)
disagreement=(test_pred==0)&guideline_positive
errors=(test_pred!=y_test)
uncertainty=1.0-2.0*np.abs(probabilities['xgboost_raw']-0.5)
review_n=max(1,int(np.ceil(len(y_test)*0.20)))
uncertain_idx=np.argsort(-uncertainty,kind='mergesort')[:review_n]
review_summary={
    'validation_threshold_selected':float(VALIDATION_THRESHOLD),
    'model_error_rate_all':float(errors.mean()),
    'ontology_flags_low_model_score_count':int(disagreement.sum()),
    'positive_cases_in_ontology_flagged_group':int(y_test[disagreement].sum()),
    'ontology_flagged_group_positive_rate':float(y_test[disagreement].mean()) if disagreement.any() else None,
    'ontology_flagged_group_model_error_rate':float(errors[disagreement].mean()) if disagreement.any() else None,
    'confidence_review_capacity':0.20,
    'confidence_review_count':int(review_n),
    'fraction_of_all_model_errors_in_confidence_review':float(errors[uncertain_idx].sum()/max(1,errors.sum())),
    'confidence_review_error_rate':float(errors[uncertain_idx].mean()),
}
print('Disagreement/review analysis (descriptive, not proof of uncertainty):',review_summary)

In [ ]:
# Research figures from actual temporal-test predictions.
figure_paths=[]
# ROC and PR curves
fig, axes = plt.subplots(1,2,figsize=(12,5))
for name in ['logistic_raw','xgboost_raw','logistic_ontology','xgboost_ontology','xgboost_ontology_no_ethnicity','xgboost_generic_control']:
    p=probabilities[name]
    fpr,tpr,_=roc_curve(y_test,p)
    prec,rec,_=precision_recall_curve(y_test,p)
    axes[0].plot(fpr,tpr,label=f"{name.replace('_',' ')} (AUC {roc_auc_score(y_test,p):.3f})")
    axes[1].plot(rec,prec,label=f"{name.replace('_',' ')} (AP {average_precision_score(y_test,p):.3f})")
axes[0].plot([0,1],[0,1],'--',color='gray',linewidth=1); axes[0].set_title('ROC curve'); axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[1].set_title('Precision–recall curve'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
for ax in axes: ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.tight_layout(); pth=ARTIFACT_DIR/'roc_pr_curves.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

# Calibration plot
fig,ax=plt.subplots(figsize=(6,5))
for name in ['logistic_raw','xgboost_raw','logistic_ontology','xgboost_ontology','xgboost_ontology_no_ethnicity','xgboost_generic_control']:
    frac,mean_pred=calibration_curve(y_test,probabilities[name],n_bins=10,strategy='quantile')
    ax.plot(mean_pred,frac,marker='o',label=name.replace('_',' '))
ax.plot([0,1],[0,1],'--',color='gray'); ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed positive fraction'); ax.set_title('Calibration on temporal test'); ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.tight_layout(); pth=ARTIFACT_DIR/'calibration.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

# Fixed screening capacity sensitivity
fig,ax=plt.subplots(figsize=(8,5))
for policy,group in screening_budget_df.groupby('policy'):
    group=group.sort_values('capacity')
    ax.plot(group['capacity']*100,group['sensitivity'],marker='o',label=policy)
ax.set_xlabel('Screening capacity (% of eligible test cohort)'); ax.set_ylabel('Sensitivity / positive cases found'); ax.set_ylim(0,1); ax.set_title('Sensitivity at fixed screening capacity'); ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.tight_layout(); pth=ARTIFACT_DIR/'capacity_sensitivity.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

# Primary effect interval
fig,ax=plt.subplots(figsize=(7,2.8))
ax.errorbar([primary_effect['difference']],[0],xerr=[[primary_effect['difference']-primary_effect['ci_low']],[primary_effect['ci_high']-primary_effect['difference']]],fmt='o',capsize=6)
ax.axvline(0,color='gray',linestyle='--'); ax.set_yticks([]); ax.set_xlabel('Hybrid minus ML-only sensitivity at 20% capacity'); ax.set_title('Paired bootstrap 95% interval'); ax.grid(axis='x',alpha=.25)
plt.tight_layout(); pth=ARTIFACT_DIR/'primary_effect_ci.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

In [ ]:
# Save reproducibility artifacts consumed by the Streamlit app and paper workflow.
# Participant-level predictions are created locally for reproducibility; review before sharing publicly.
output_predictions = pd.DataFrame({
    'target': y_test,
    'probability_logistic_raw': probabilities['logistic_raw'],
    'probability_xgboost_raw': probabilities['xgboost_raw'],
    'probability_logistic_ontology': probabilities['logistic_ontology'],
    'probability_xgboost_ontology': probabilities['xgboost_ontology'],
    'probability_xgboost_ontology_no_ethnicity': probabilities['xgboost_ontology_no_ethnicity'],
    'probability_xgboost_generic_control': probabilities['xgboost_generic_control'],
    'ontology_rule_priority_score': test_rule_score,
    'ontology_screening_eligible': test_eligible,
    'hybrid_priority_score': hybrid_score,
    'hybrid_priority_score_no_higher_risk_ethnicity_flag': hybrid_score_no_ethnicity,
    'ontology_priority_score_no_higher_risk_ethnicity_flag': test_rule_score_no_ethnicity,
    'ml_rule_disagreement_low_score': disagreement.astype(int),
})
output_predictions.to_csv(ARTIFACT_DIR/'test_predictions.csv',index=False)

metric_results={}
for name, detail in metrics_by_model.items():
    ens=detail['ensemble']
    metric_results[name]={
        'auroc_mean':detail.get('auroc_mean'),'auroc_sd':detail.get('auroc_sd'),
        'auprc_mean':detail.get('auprc_mean'),'auprc_sd':detail.get('auprc_sd'),
        'brier_mean':detail.get('brier_mean'),'brier_sd':detail.get('brier_sd'),
        'ensemble_auroc':ens['auroc'],'ensemble_auprc':ens['auprc'],'ensemble_brier':ens['brier'],
        'n':ens['n'],'positive_count':ens['positive_count'],'prevalence':ens['prevalence'],
        'best_parameters':detail['cv_best_params'],
    }

results_payload={
    'project':'ClinOntoScreen',
    'study_title':'Ontology-Guided Screening for Undetected Dysglycemia: Evaluating Clinical Knowledge Integration Under Temporal Shift in NHANES',
    'generated_at_utc':datetime.now(timezone.utc).isoformat(),
    'metadata':{
        'python_version':platform.python_version(),'pandas_version':pd.__version__,
        'sklearn_version':__import__('sklearn').__version__,'xgboost_version':__import__('xgboost').__version__,
        'seeds':SEEDS,'primary_capacity':PRIMARY_CAPACITY,'capacities':CAPACITIES,'bootstrap_replicates':N_BOOTSTRAP,
        'historical_cycle':'NHANES 2017–March 2020 pre-pandemic','temporal_test_cycle':'NHANES August 2021–August 2023',
        'outcome':'LBXGH >= 5.7 among adults reporting neither prior diabetes nor prior prediabetes',
        'outcome_warning':'Single-measurement research label; not a confirmed clinical diagnosis.',
        'ontology_file':'diabetes_screening.ttl','hybrid_alpha':HYBRID_ALPHA,'hybrid_alpha_no_higher_risk_ethnicity_flag':HYBRID_ALPHA_NO_ETHNICITY,'validation_threshold_for_descriptive_disagreement':VALIDATION_THRESHOLD,
        'test_set_used_for_tuning':False,
        'protocol_tag':'protocol-v1','git_commit':os.environ.get('GIT_COMMIT','unknown'),'git_tag':os.environ.get('GIT_TAG','unknown'),
    },
    'cohort':{
        'historical_n':int(len(historical)),'historical_positive_count':int(historical['target'].sum()),'historical_prevalence':float(historical['target'].mean()),
        'test_n':int(len(temporal_test)),'test_positive_count':int(y_test.sum()),'test_prevalence':float(y_test.mean()),
        'test_diabetes_range_secondary_positive_count':int(temporal_test['target_diabetes_range'].sum()),
    },
    'metrics':metric_results,
    'screening_budgets':screening_budget_rows,
    'primary_effect':primary_effect,
    'race_ethnicity_ablation':{'interpretation':'Removes the higher-risk race/ethnicity risk-factor flag from the screening rule; retains the Asian-specific BMI threshold and raw race/ethnicity predictor. This is not a fully race-blind model.', 'primary_effect_without_higher_risk_ethnicity_flag':primary_effect_no_ethnicity, 'alpha_validation_table_without_higher_risk_ethnicity_flag':alpha_results_no_ethnicity},
    'review_analysis':review_summary,
    'alpha_validation_table':alpha_results,
    'source_reports':{'historical':historical_report,'temporal_test':temporal_report},
    'feature_columns':{'raw':RAW_FEATURES,'ontology':ONTOLOGY_FEATURES,'manual_control':MANUAL_FEATURES},
    'figures':[Path(x).name for x in figure_paths],
}
(ARTIFACT_DIR/'research_results.json').write_text(json.dumps(results_payload,indent=2,default=lambda x: None if pd.isna(x) else str(x)),encoding='utf-8')
(ARTIFACT_DIR/'feature_manifest.json').write_text(json.dumps({'raw_features':RAW_FEATURES,'ontology_features':ONTOLOGY_FEATURES,'manual_control_features':MANUAL_FEATURES,'forbidden_predictors':['LBXGH','DIQ010','DIQ160','DIQ050','DIQ070','SEQN','glucose','hba1c','diabetes medication'], 'selected_model_inputs':'No target-source variables may be passed to any model.'},indent=2),encoding='utf-8')

bundle={
    'project':'ClinOntoScreen','models':models,
    'raw_feature_columns':RAW_FEATURES,'ontology_feature_columns':ONTOLOGY_FEATURES,
    'manual_feature_columns':MANUAL_FEATURES,'hybrid_alpha':HYBRID_ALPHA,
    'validation_threshold':VALIDATION_THRESHOLD,'seeds':SEEDS,
    'model_metadata':{name:{'family':'logistic' if name.startswith('logistic') else 'xgboost','params':best_params[name]} for name in models},
}
joblib.dump(bundle,ARTIFACT_DIR/'model_bundle.joblib',compress=3)

run_manifest={
  'generated_at_utc':datetime.now(timezone.utc).isoformat(),
  'data_source_urls':[cfg['base_url'] for cfg in CYCLE_CONFIGS.values()],
  'sources':{'historical':historical_report,'temporal_test':temporal_report},
  'parameters':results_payload['metadata'],'artifacts':[p.name for p in ARTIFACT_DIR.iterdir() if p.is_file()]
}
(ARTIFACT_DIR/'run_manifest.json').write_text(json.dumps(run_manifest,indent=2,default=str),encoding='utf-8')
zip_path=WORK_DIR/'clinontoscreen_artifacts.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in ARTIFACT_DIR.rglob('*'):
        if path.is_file(): z.write(path,arcname=f'artifacts/{path.relative_to(ARTIFACT_DIR)}')
print('Saved artifacts:')
for p in sorted(ARTIFACT_DIR.iterdir()):
    if p.is_file(): print(f'  {p.name}: {p.stat().st_size:,} bytes')
print('ZIP:',zip_path)
print('\nPrimary endpoint summary:')
display(pd.DataFrame([primary_effect]))

# Download the results bundle in Colab; in Kaggle the zip appears under /kaggle/working.
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except Exception:
    print('If running in Kaggle, use the Output/Files panel to download:', zip_path)


## How to use the generated artifacts

- `model_bundle.joblib`: ensemble model pipelines and the tuned hybrid-policy coefficient. Load only from a trusted source because joblib/pickle formats can execute code when loaded.
- `research_results.json`: metrics, confidence intervals, cohort counts, and figure list used by the dashboard.
- `test_predictions.csv`: participant-level prediction outputs without participant IDs. Keep private by default and review sharing requirements before publication.
- `feature_manifest.json`: raw, ontology and manual-control features plus leakage exclusions.
- `data_qc.json` and `run_manifest.json`: source-file availability, missingness, package versions and run metadata.
- `*.png`: paper/dashboard figures generated by this run.

To populate the app, copy the generated `artifacts/` directory into the project repository's `artifacts/` folder locally. Keep large model files and participant-level predictions out of Git unless there is a deliberate, approved release plan. Run `streamlit run app.py` from the project root.